### IMPORT

In [1]:
!pip install fastdtw

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.4/133.4 kB 3.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for fastdtw: filename=fastdtw-0.3.4-cp312-cp312-linux_x86_64.whl size=567858 sha256=9476b3e730b7f0298e1732bb2dd1bac23decfe95fc10ddee3e619058b27e9527
  Stored in directory: /root/.cache/pip/wheels/ab/d0/26/b82cb0f49ae73e5e6bba4e8462fff2c9851d7bd2ec64f8891e
Successfully built fastdtw


In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
import warnings
warnings.filterwarnings("ignore")

In [4]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from fastdtw import fastdtw
from scipy.special import comb
from scipy import stats, signal
from scipy.interpolate import interp1d
from scipy.spatial.distance import euclidean

from itertools import combinations
from IPython.display import display

In [5]:
NOSE = 0
LEFT_EYE = 1
RIGHT_EYE = 2
LEFT_EAR = 3
RIGHT_EAR = 4
LEFT_SHOULDER = 5
RIGHT_SHOULDER = 6
LEFT_ELBOW = 7
RIGHT_ELBOW = 8
LEFT_WRIST = 9
RIGHT_WRIST = 10
LEFT_HIP = 11
RIGHT_HIP = 12
LEFT_KNEE = 13
RIGHT_KNEE = 14
LEFT_ANKLE = 15
RIGHT_ANKLE = 16

In [15]:
def loadSequences(root):
    seqs = []
    for dirpath, _, filenames in os.walk(root):
        for f in filenames:
            if not (f := f.lower()).endswith(".npy"):
                continue
            if f.startswith("unlabeled"):
                continue
            match f:
                case name if "curveball" in name:
                    label = "curveball"
                case name if "cutter" in name:
                    label = "cutter"
                case name if "four_seam_fastball" in name:
                    label = "four_seam_fastball"
                case name if "slider" in name:
                    label = "slider"
                case name if "splitter" in name:
                    label = "splitter"
                case name if "two_seam_fastball" in name:
                    label = "two_seam_fastball"
                case name if "changeup" in name:
                    label = "changeup"
                case name if "fastball" in name:
                    label = "fastball"
                case _:
                    continue
            seqs.append((np.load(os.path.join(dirpath, f), allow_pickle=True), label))
    return seqs


### FEATURE EXTRACTION

In [2]:
# ----------------------------------------------------------------------
# Helper functions
# ----------------------------------------------------------------------
def _time_to_peak(series):
    """Normalized time (0-1) of maximum value."""
    if len(series) == 0:
        return np.nan
    return np.argmax(series) / len(series)

def _time_to_min(series):
    """Normalized time (0-1) of minimum value."""
    if len(series) == 0:
        return np.nan
    return np.argmin(series) / len(series)

def _velocity(series, dt):
    """First derivative using central differences."""
    return np.gradient(series, dt)

def _acceleration(series, dt):
    """Second derivative."""
    return np.gradient(np.gradient(series, dt), dt)

def _angular_velocity(angle_series, dt):
    """Angular velocity in degrees per second, handling wraparound."""
    # Unwrap to avoid jumps near 360->0
    unwrapped = np.unwrap(np.deg2rad(angle_series)) * 180 / np.pi
    return np.gradient(unwrapped, dt)

def _path_length(trajectory):
    """Total Euclidean distance traveled along a trajectory."""
    return np.sum(np.linalg.norm(np.diff(trajectory, axis=0), axis=1))

def _angle_between_points(p1, p2, p3):
    """Angle at p2 formed by p1-p2-p3 (degrees)."""
    a = np.array(p1)
    b = np.array(p2)
    c = np.array(p3)
    ba = a - b
    bc = c - b
    cosine = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc) + 1e-8)
    return np.degrees(np.arccos(np.clip(cosine, -1.0, 1.0)))

def _forearm_angle(shoulder, elbow, wrist, reference='vertical'):
    """
    Angle of the forearm (elbow->wrist) relative to vertical or horizontal.
    Returns angle in degrees [0,180]. For vertical: 0 = straight down, 180 = straight up.
    For horizontal: 0 = pointing right (if positive x), 180 = left.
    """
    forearm = np.array(wrist) - np.array(elbow)
    if reference == 'vertical':
        ref_vec = np.array([0, -1])   # downward (assuming y increases downward)
    elif reference == 'horizontal':
        ref_vec = np.array([1, 0])    # right
    else:
        raise ValueError("reference must be 'vertical' or 'horizontal'")
    dot = np.dot(forearm, ref_vec)
    norm_prod = np.linalg.norm(forearm) * np.linalg.norm(ref_vec) + 1e-8
    angle = np.degrees(np.arccos(np.clip(dot / norm_prod, -1.0, 1.0)))
    return angle

def _compute_angles(frame):
    """Compute all joint angles and orientations for one frame."""
    angles = {}

    # Arm angles
    angles['left_shoulder'] = _angle_between_points(
        frame[LEFT_HIP], frame[LEFT_SHOULDER], frame[LEFT_ELBOW])
    angles['left_elbow'] = _angle_between_points(
        frame[LEFT_SHOULDER], frame[LEFT_ELBOW], frame[LEFT_WRIST])
    angles['right_shoulder'] = _angle_between_points(
        frame[RIGHT_HIP], frame[RIGHT_SHOULDER], frame[RIGHT_ELBOW])
    angles['right_elbow'] = _angle_between_points(
        frame[RIGHT_SHOULDER], frame[RIGHT_ELBOW], frame[RIGHT_WRIST])

    # Leg angles
    angles['left_hip'] = _angle_between_points(
        frame[LEFT_SHOULDER], frame[LEFT_HIP], frame[LEFT_KNEE])
    angles['left_knee'] = _angle_between_points(
        frame[LEFT_HIP], frame[LEFT_KNEE], frame[LEFT_ANKLE])
    angles['right_hip'] = _angle_between_points(
        frame[RIGHT_SHOULDER], frame[RIGHT_HIP], frame[RIGHT_KNEE])
    angles['right_knee'] = _angle_between_points(
        frame[RIGHT_HIP], frame[RIGHT_KNEE], frame[RIGHT_ANKLE])

    # Trunk inclination (angle of spine relative to vertical)
    shoulder_mid = (frame[LEFT_SHOULDER] + frame[RIGHT_SHOULDER]) / 2
    hip_mid = (frame[LEFT_HIP] + frame[RIGHT_HIP]) / 2
    trunk_vec = shoulder_mid - hip_mid
    vertical = np.array([0, -1])
    angles['trunk_inclination'] = np.degrees(np.arccos(
        np.dot(trunk_vec, vertical) / (np.linalg.norm(trunk_vec) + 1e-8)))

    # Pelvis rotation (angle of hip line relative to horizontal)
    hip_line = frame[RIGHT_HIP] - frame[LEFT_HIP]
    angles['pelvis_rotation'] = np.degrees(np.arctan2(hip_line[1], hip_line[0]))

    # Shoulder rotation (angle of shoulder line relative to horizontal)
    shoulder_line = frame[RIGHT_SHOULDER] - frame[LEFT_SHOULDER]
    angles['shoulder_rotation'] = np.degrees(np.arctan2(shoulder_line[1], shoulder_line[0]))

    # Hip-shoulder separation (X-factor) = difference in rotation angles
    angles['hip_shoulder_sep'] = angles['shoulder_rotation'] - angles['pelvis_rotation']

    # Head tilt (angle of ear line)
    ear_line = frame[RIGHT_EAR] - frame[LEFT_EAR] if RIGHT_EAR in frame and LEFT_EAR in frame else np.array([0, 0])
    angles['head_tilt'] = np.degrees(np.arctan2(ear_line[1], ear_line[0]))

    # Forearm angle relative to vertical (right arm) – proxy for arm slot
    angles['right_forearm_vertical'] = _forearm_angle(
        frame[RIGHT_SHOULDER], frame[RIGHT_ELBOW], frame[RIGHT_WRIST], 'vertical')

    # Wrist angle relative to forearm (angle between forearm and hand)
    # Hand vector approximated by wrist to something? Not available. Skip.

    return angles

# ----------------------------------------------------------------------
# Main feature extraction function
# ----------------------------------------------------------------------
def extractFeatures(seq, class_label, fps=60):
    if len(seq) > 0 and isinstance(seq[0], dict):
        glove_seq = np.array([frame.get("glove_box", [-1, -1, -1, -1]) for frame in seq], dtype=np.float32)
        seq = np.array([frame["keypoints"] for frame in seq], dtype=np.float32)
    else:
        glove_seq = np.full((len(seq), 4), -1.0, dtype=np.float32)
    """
    Extract kinematic features from a pose sequence (T x 17 x 2).
    """
    T = seq.shape[0]
    dt = 1.0 / fps
    duration = T * dt

    # Compute angles for each frame
    angles_list = [_compute_angles(seq[t]) for t in range(T)]
    angle_names = angles_list[0].keys()
    angles = {name: np.array([a[name] for a in angles_list], dtype=float)
              for name in angle_names}

    # Position data for keypoints
    pos = {
        'left_knee': seq[:, LEFT_KNEE, :],
        'right_knee': seq[:, RIGHT_KNEE, :],
        'left_ankle': seq[:, LEFT_ANKLE, :],
        'right_ankle': seq[:, RIGHT_ANKLE, :],
        'left_shoulder': seq[:, LEFT_SHOULDER, :],
        'right_shoulder': seq[:, RIGHT_SHOULDER, :],
        'left_hip': seq[:, LEFT_HIP, :],
        'right_hip': seq[:, RIGHT_HIP, :],
        'left_wrist': seq[:, LEFT_WRIST, :],
        'right_wrist': seq[:, RIGHT_WRIST, :],
        'left_elbow': seq[:, LEFT_ELBOW, :],
        'right_elbow': seq[:, RIGHT_ELBOW, :],
    }

    # Torso length for normalization (using first frame)
    shoulder_mid = (seq[0, LEFT_SHOULDER] + seq[0, RIGHT_SHOULDER]) / 2
    hip_mid = (seq[0, LEFT_HIP] + seq[0, RIGHT_HIP]) / 2
    torso_length = euclidean(shoulder_mid, hip_mid)

    features = {'class': class_label, 'duration': duration}

    # ------------------------------------------------------------------
    # 1. Key angle statistics (only mean, std, range for now)
    # ------------------------------------------------------------------
    for name, series in angles.items():
        series_clean = series[~np.isnan(series)]
        if len(series_clean) == 0:
            continue
        features[f'{name}_mean'] = np.mean(series_clean)
        features[f'{name}_std'] = np.std(series_clean)
        features[f'{name}_range'] = np.nanmax(series) - np.nanmin(series)

    # ------------------------------------------------------------------
    # 2. Temporal features (timing of peaks / minima)
    # ------------------------------------------------------------------
    # Left knee height (y-coordinate)
    left_knee_y = pos['left_knee'][:, 1]
    features['time_to_peak_left_knee'] = _time_to_peak(left_knee_y)

    # Right elbow angle
    if 'right_elbow' in angles:
        features['time_to_min_right_elbow'] = _time_to_min(angles['right_elbow'])
        features['time_to_max_right_elbow'] = _time_to_peak(angles['right_elbow'])

    # Left elbow angle
    if 'left_elbow' in angles:
        features['time_to_min_left_elbow'] = _time_to_min(angles['left_elbow'])
        features['time_to_max_left_elbow'] = _time_to_peak(angles['left_elbow'])

    # Hip-shoulder separation
    if 'hip_shoulder_sep' in angles:
        features['time_to_max_hip_shoulder_sep'] = _time_to_peak(angles['hip_shoulder_sep'])
        features['time_to_min_hip_shoulder_sep'] = _time_to_min(angles['hip_shoulder_sep'])

    # Right forearm vertical angle
    if 'right_forearm_vertical' in angles:
        features['time_to_min_forearm_angle'] = _time_to_min(angles['right_forearm_vertical'])
        features['time_to_max_forearm_angle'] = _time_to_peak(angles['right_forearm_vertical'])

    # ------------------------------------------------------------------
    # 3. Velocities, accelerations, and peaks
    # ------------------------------------------------------------------
    # Angular velocities for key joints
    for joint in ['right_elbow', 'left_elbow', 'right_shoulder', 'left_shoulder',
                  'right_knee', 'left_knee', 'hip_shoulder_sep', 'pelvis_rotation',
                  'shoulder_rotation']:
        if joint in angles:
            ang_vel = _angular_velocity(angles[joint], dt)
            features[f'{joint}_ang_vel_max'] = np.nanmax(np.abs(ang_vel))
            features[f'{joint}_ang_vel_mean'] = np.nanmean(np.abs(ang_vel))
            features[f'{joint}_time_to_max_ang_vel'] = _time_to_peak(np.abs(ang_vel))

    # Linear velocity of right wrist
    rw_x, rw_y = pos['right_wrist'][:, 0], pos['right_wrist'][:, 1]
    vel_x = _velocity(rw_x, dt)
    vel_y = _velocity(rw_y, dt)
    speed_rw = np.sqrt(vel_x**2 + vel_y**2)
    features['right_wrist_speed_max'] = np.max(speed_rw)
    features['right_wrist_speed_mean'] = np.mean(speed_rw)
    features['right_wrist_time_to_max_speed'] = _time_to_peak(speed_rw)

    # Linear velocity of left knee
    lk_x, lk_y = pos['left_knee'][:, 0], pos['left_knee'][:, 1]
    vel_x_lk = _velocity(lk_x, dt)
    vel_y_lk = _velocity(lk_y, dt)
    speed_lk = np.sqrt(vel_x_lk**2 + vel_y_lk**2)
    features['left_knee_speed_max'] = np.max(speed_lk)
    features['left_knee_speed_mean'] = np.mean(speed_lk)
    features['left_knee_time_to_max_speed'] = _time_to_peak(speed_lk)

    # ------------------------------------------------------------------
    # 4. Key positions and distances
    # ------------------------------------------------------------------
    # Maximum left knee height
    features['left_knee_max_height'] = np.max(left_knee_y)

    # Stride width at end (normalized by torso length)
    stride = np.linalg.norm(pos['left_ankle'][-1] - pos['right_ankle'][-1])
    features['stride_width_norm'] = stride / torso_length

    # Minimum distance from right wrist to right shoulder (proxy for hand near shoulder)
    dist_rw_rs = np.linalg.norm(pos['right_wrist'] - pos['right_shoulder'], axis=1)
    features['min_dist_wrist_shoulder_norm'] = np.min(dist_rw_rs) / torso_length

    # Forward displacement of left knee (relative to start)
    left_knee_start_x = pos['left_knee'][0, 0]
    left_knee_forward = pos['left_knee'][:, 0] - left_knee_start_x
    features['left_knee_max_forward_norm'] = np.max(left_knee_forward) / torso_length

    # Center of mass (centroid) range
    centroid = np.nanmean(seq, axis=1)  # (T,2)
    features['com_vertical_range_norm'] = (np.max(centroid[:,1]) - np.min(centroid[:,1])) / torso_length
    features['com_horizontal_range_norm'] = (np.max(centroid[:,0]) - np.min(centroid[:,0])) / torso_length
    features['com_final_vertical_norm'] = centroid[-1, 1] / torso_length
    features['com_final_horizontal_norm'] = centroid[-1, 0] / torso_length

    # ------------------------------------------------------------------
    # 5. Trajectory shape
    # ------------------------------------------------------------------
    features['left_knee_path_length_norm'] = _path_length(pos['left_knee']) / torso_length
    features['right_wrist_path_length_norm'] = _path_length(pos['right_wrist']) / torso_length

    # ------------------------------------------------------------------
    # 6. Symmetry / asymmetry at peak left knee
    # ------------------------------------------------------------------
    peak_idx = np.argmax(left_knee_y)
    for side in ['shoulder', 'elbow', 'hip', 'knee']:
        left_name = f'left_{side}'
        right_name = f'right_{side}'
        if left_name in angles and right_name in angles:
            features[f'{side}_angle_diff_at_peak'] = angles[left_name][peak_idx] - angles[right_name][peak_idx]

    # Range ratios (left/right) for symmetry
    for side in ['shoulder', 'elbow', 'hip', 'knee']:
        left_name = f'left_{side}'
        right_name = f'right_{side}'
        if left_name in angles and right_name in angles:
            left_range = np.nanmax(angles[left_name]) - np.nanmin(angles[left_name])
            right_range = np.nanmax(angles[right_name]) - np.nanmin(angles[right_name])
            if right_range != 0:
                features[f'{side}_range_ratio'] = left_range / right_range
            else:
                features[f'{side}_range_ratio'] = np.nan

    # ------------------------------------------------------------------
    # 7. Correlations and coordination
    # ------------------------------------------------------------------
    # Correlation between left knee height and right elbow angle
    if 'right_elbow' in angles:
        # Use Pearson correlation (ignoring phase)
        pearson = np.corrcoef(left_knee_y, angles['right_elbow'])[0, 1]
        features['corr_left_knee_right_elbow'] = pearson
        # Also compute cross-correlation max to capture lagged relationship
        corr = signal.correlate(left_knee_y - np.mean(left_knee_y),
                                angles['right_elbow'] - np.mean(angles['right_elbow']),
                                mode='full')
        features['max_cross_corr_left_knee_right_elbow'] = np.max(np.abs(corr))

    # Correlation between left knee height and hip-shoulder separation
    if 'hip_shoulder_sep' in angles:
        pearson = np.corrcoef(left_knee_y, angles['hip_shoulder_sep'])[0, 1]
        features['corr_left_knee_hip_shoulder_sep'] = pearson

    # ------------------------------------------------------------------
    # 8. Additional timing relationships
    # ------------------------------------------------------------------
    # Time difference between peak knee height and peak elbow flexion
    if 'right_elbow' in angles:
        t_knee = _time_to_peak(left_knee_y)
        t_elbow_min = _time_to_min(angles['right_elbow'])  # min angle = flexion
        features['time_diff_knee_peak_elbow_min'] = t_knee - t_elbow_min

    # Time difference between peak knee height and max hip-shoulder separation
    if 'hip_shoulder_sep' in angles:
        t_hip_sep_max = _time_to_peak(angles['hip_shoulder_sep'])
        features['time_diff_knee_peak_hip_sep_max'] = t_knee - t_hip_sep_max

    valid_glove = glove_seq[:, 0] != -1
    if np.sum(valid_glove) > 2:
        glove_seq_interp = np.copy(glove_seq)
        for i in range(4):
            arr = glove_seq_interp[:, i]
            arr[~valid_glove] = np.interp(np.arange(len(arr))[~valid_glove], np.arange(len(arr))[valid_glove], arr[valid_glove])

        glove_centers = np.stack([(glove_seq_interp[:, 0] + glove_seq_interp[:, 2]) / 2,
                                  (glove_seq_interp[:, 1] + glove_seq_interp[:, 3]) / 2], axis=1)

        features['glove_max_height_norm'] = np.max(glove_centers[:, 1]) / torso_length
        features['glove_min_height_norm'] = np.min(glove_centers[:, 1]) / torso_length
        features['glove_height_range_norm'] = (np.max(glove_centers[:, 1]) - np.min(glove_centers[:, 1])) / torso_length
        g_vel_x = _velocity(glove_centers[:, 0], dt)
        g_vel_y = _velocity(glove_centers[:, 1], dt)
        g_speed = np.sqrt(g_vel_x**2 + g_vel_y**2)
        features['glove_speed_max'] = np.max(g_speed)
        features['glove_speed_mean'] = np.mean(g_speed)
        features['glove_time_to_max_speed'] = _time_to_peak(g_speed)
        shoulder_mids = (seq[:, LEFT_SHOULDER] + seq[:, RIGHT_SHOULDER]) / 2
        dist_to_shoulder = np.linalg.norm(glove_centers - shoulder_mids, axis=1)
        features['glove_dist_to_shoulder_min_norm'] = np.min(dist_to_shoulder) / torso_length
        features['glove_dist_to_shoulder_mean_norm'] = np.mean(dist_to_shoulder) / torso_length

    return features

### DYNAMIC TIME WARPING

In [8]:
def dtw_distance(seq1, seq2):
    if len(seq1) > 0 and isinstance(seq1[0], dict):
        seq1 = np.array([frame["keypoints"] for frame in seq1], dtype=np.float32)
    if len(seq2) > 0 and isinstance(seq2[0], dict):
        seq2 = np.array([frame["keypoints"] for frame in seq2], dtype=np.float32)
    """
    Compute DTW distance between two pose sequences.
    seq1, seq2: arrays of shape (T, 17, 2)
    Returns: DTW distance (float)
    """
    # Flatten each frame into a vector of length 34 (17 keypoints * 2 coordinates)
    flat1 = seq1.reshape(seq1.shape[0], -1)
    flat2 = seq2.reshape(seq2.shape[0], -1)
    # Use fastdtw with Euclidean distance
    distance, _ = fastdtw(flat1, flat2, dist=euclidean)
    return distance

def add_dtw_features(seqs, labels, features_df):
    """
    For each sequence, compute mean DTW distance to sequences of each class.
    Append these as new columns to features_df.
    """
    n = len(seqs)
    class_names = list(set(labels))
    # Precompute all pairwise DTW distances (n x n matrix)
    dtw_matrix = np.zeros((n, n))
    for i in range(n):
        for j in range(i+1, n):
            d = dtw_distance(seqs[i][0], seqs[j][0])  # seqs[i] is (seq, label)
            dtw_matrix[i, j] = d
            dtw_matrix[j, i] = d

    # For each class, compute mean distance from each sequence to that class
    for target_class in class_names:
        col_name = f"dtw_dist_to_{target_class}"
        distances = []
        for i in range(n):
            # Indices of all sequences with class == target_class, excluding i
            class_indices = [j for j, (_, lbl) in enumerate(seqs) if lbl == target_class and j != i]
            if class_indices:
                mean_dist = np.mean(dtw_matrix[i, class_indices])
            else:
                mean_dist = np.nan  # no other samples in that class
            distances.append(mean_dist)
        features_df[col_name] = distances

    return features_df, dtw_matrix

### STATISTICAL TESTING

In [9]:
def pairwiseTests(df, feature_cols, class_col="class", alpha=0.05):
    classes = df[class_col].unique()
    pairs = list(combinations(classes, 2))
    res = []

    for feature in feature_cols:
        for c1, c2 in pairs:
            data1 = df[df[class_col] == c1][feature].dropna()
            data2 = df[df[class_col] == c2][feature].dropna()
            if len(data1) < 2 or len(data2) < 2:
                continue
            U, p = stats.mannwhitneyu(data1, data2, alternative="two-sided")
            n1, n2 = len(data1), len(data2)
            effect_size = U / (n1 * n2)
            res.append({
                "feature": feature,
                "class1": c1,
                "class2": c2,
                "U": U,
                "p": p,
                "effect_size": effect_size,
                "n1": n1,
                "n2": n2
            })
    df = pd.DataFrame(res)
    df["significant"] = df["p"] < alpha / comb(len(classes), 2, exact=True)
    return df

### EMPIRICAL RESULTS (109, 13)

In [16]:
seqs = loadSequences("/content/drive/My Drive/Pitch-Tipping/Keypoints")
features = pd.DataFrame([extractFeatures(seq, label) for seq, label in seqs])
features, dtw_mat = add_dtw_features(seqs, [label for _, label in seqs], features)
featureCols = [col for col in features.columns if col != "class"]
display(pairwiseTests(features, featureCols).sort_values(
    ["effect_size", "p"],
    key=lambda x: (x - 0.5).abs() if x.name == "effect_size" else x,
    ascending=[False, True]
)[:10])

,feature,class1,class2,U,p,effect_size,n1,n2,significant
1290,stride_width_norm,curveball,cutter,25.0,0.007937,1.0,5,5,False
1649,glove_min_height_norm,slider,splitter,24.0,0.012121,1.0,3,8,False
1724,glove_dist_to_shoulder_min_norm,slider,splitter,24.0,0.012121,1.0,3,8,False
1814,dtw_dist_to_curveball,slider,splitter,24.0,0.012121,1.0,3,8,False
648,time_to_peak_left_knee,curveball,slider,15.0,0.015959,1.0,5,3,False
1540,knee_range_ratio,four_seam_fastball,slider,0.0,0.023810,0.0,6,3,False
865,left_elbow_time_to_max_ang_vel,four_seam_fastball,slider,0.0,0.025601,0.0,6,3,False
775,time_to_max_forearm_angle,four_seam_fastball,slider,18.0,0.026883,1.0,6,3,False
18,left_shoulder_mean,curveball,slider,0.0,0.035714,0.0,5,3,False
108,right_shoulder_mean,curveball,slider,0.0,0.035714,0.0,5,3,False


In [17]:
relabel = dict.fromkeys(["cutter", "four_seam_fastball", "two_seam_fastball"], "fastball")
relabel.update(dict.fromkeys(["curveball", "slider", "splitter"], "offspeed"))
seqs = loadSequences("/content/drive/My Drive/Pitch-Tipping/Keypoints")
seqs = list(map(lambda pair: (pair[0], relabel[pair[1]]), seqs))
features = pd.DataFrame([extractFeatures(seq, label) for seq, label in seqs])
features, dtw_mat = add_dtw_features(seqs, [label for _, label in seqs], features)
featureCols = [col for col in features.columns if col != "class"]
display(pairwiseTests(features, featureCols).sort_values(
    ["effect_size", "p"],
    key=lambda x: (x - 0.5).abs() if x.name == "effect_size" else x,
    ascending=[False, True]
)[:10])

,feature,class1,class2,U,p,effect_size,n1,n2,significant
40,right_forearm_vertical_mean,offspeed,fastball,161.0,0.013224,0.774038,16,13,True
85,left_knee_max_height,offspeed,fastball,56.0,0.037250,0.269231,16,13,True
82,left_knee_speed_max,offspeed,fastball,61.0,0.062357,0.293269,16,13,False
113,glove_time_to_max_speed,offspeed,fastball,144.0,0.080354,0.692308,16,13,False
1,left_shoulder_mean,offspeed,fastball,64.0,0.083240,0.307692,16,13,False
10,right_elbow_mean,offspeed,fastball,142.0,0.100076,0.682692,16,13,False
27,trunk_inclination_range,offspeed,fastball,68.0,0.119523,0.326923,16,13,False
57,left_elbow_time_to_max_ang_vel,offspeed,fastball,138.0,0.139304,0.663462,16,13,False
83,left_knee_speed_mean,offspeed,fastball,70.0,0.141813,0.336538,16,13,False
19,right_hip_mean,offspeed,fastball,137.0,0.154094,0.658654,16,13,False


### LEVEL 3: MACHINE LEARNING ENGINES (FASTBALL VS OFFSPEED)
*N=40 Safe Models: Random Forest and L1 Lasso*

In [19]:
# --- BINARY CLASSIFICATION (FASTBALL VS OFFSPEED) ---
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import numpy as np

# 1. Prepare Binary Labels
df_ml = features.copy()

# Group Pitch Types for N=40 robustness
fastballs = ["four_seam_fastball", "two_seam_fastball", "cutter"]
offspeeds = ["curveball", "slider", "splitter", "changeup"]


# Overriding arrays to guarantee mapping safety
safe_fastballs = ["fastball", "cutter", "four_seam_fastball", "two_seam_fastball"]
safe_offspeeds = ["offspeed", "changeup", "curveball", "slider", "splitter"]

def label_bin(val):
    s = str(val).lower().strip()
    if any(fb in s for fb in safe_fastballs): return 0
    if any(osp in s for osp in safe_offspeeds): return 1
    return -1

df_ml['binary_class'] = df_ml['class'].apply(label_bin)
print(f"\n🔍 Raw Labels Found in memory: {list(df_ml['class'].unique())}")
print(f"🔢 Mapped Binary Classes: {dict(df_ml['binary_class'].value_counts())}")


df_ml = df_ml[df_ml['binary_class'] != -1] # Drop unrecognized pitches

# 2. Prepare Feature Matrix X and Target y
X_raw = df_ml.drop(columns=['class', 'binary_class'])
# Drop columns that are entirely NaN or have too many missing values
X_clean = X_raw.dropna(axis=1, thresh=int(len(X_raw)*0.8))
X_clean = X_clean.fillna(X_clean.median()) # Impute remaining safely
y = df_ml['binary_class']

print(f"Dataset ready. Fastballs: {sum(y==0)}, Offspeed: {sum(y==1)}, Total Features: {X_clean.shape[1]}")

feature_names = X_clean.columns

# --------------------------------------------------------
# 3. Model 1: Random Forest
# --------------------------------------------------------
print("\n--- Model 1: Random Forest Classifier ---")
rf = RandomForestClassifier(n_estimators=100, max_depth=3, random_state=42, class_weight="balanced")
rf.fit(X_clean, y)

# Get Top 5 Feature Importances
importances = rf.feature_importances_
top_indices = np.argsort(importances)[::-1][:5]

print("🚨 TOP 5 KINEMATIC TELLS (Random Forest):")
for i in top_indices:
    print(f"{feature_names[i]}: {importances[i]*100:.1f}% AI Importance")

# --------------------------------------------------------
# 4. Model 2: Logistic Regression (L1 Penalty / Lasso)
# --------------------------------------------------------
print("\n--- Model 2: Logistic Regression (L1 Lasso) ---")
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_clean)

lr = LogisticRegression(penalty='l1', solver='liblinear', C=0.5, random_state=42, class_weight="balanced")
lr.fit(X_scaled, y)

# The L1 penalty mathematically forces useless features to exactly 0.0
coefs = lr.coef_[0]
nonzero_indices = np.where(coefs != 0)[0]
sorted_nonzero = sorted(nonzero_indices, key=lambda i: abs(coefs[i]), reverse=True)

print(f"L1 Lasso mathematically deleted {len(feature_names) - len(nonzero_indices)} random noisy features!")
print("🚨 TOP COMMANDING KINEMATIC TELLS (L1 Logistic Regression):")
for i in sorted_nonzero[:5]:
    direction = "HIGHER" if coefs[i] > 0 else "LOWER"
    print(f"{feature_names[i]}: Coefficient Magnitude {abs(coefs[i]):.3f} (Mechanic is numerically {direction} on Offspeed pitches)")

print("\n🎯 >> Type the highest ranking Feature Name directly into Dashboard Tab 2 -> Level 4 AI Highlighter! <<")


Dataset ready. Fastballs: 0, Offspeed: 0, Total Features: 118

--- Model 1: Random Forest Classifier ---


ValueError: Found array with 0 sample(s) (shape=(0, 118)) while a minimum of 1 is required by RandomForestClassifier.